# 第 3 章 MATLAB 编程与傅里叶变换 · 代码实战（Python）

> 本书章代码的 Python 实现，逐段中文注释；对应 MATLAB 版本见 `matlab/fourier/ch3/`。

## 卷积 · example

`ch3_conv_example.py`

In [ ]:
# ===== 书 3.6 节 + 习题 3.2：两个高斯函数的 FFT 卷积与解析对比 =====
# 参数（书 3.6 节）：wa = 0.3 m、wb = 0.2 m——注释原文 "[exp(-pi) radius]"，
# 意思是 fa(wa) = exp(-pi) ≈ 0.043 的半径（不是 1/e 半径！）；L = 2、M = 200。
# 支撑判据（式 2.27）：Da ≈ 0.9、Db ≈ 0.6，和 1.5 < L = 2 -> 卷绕可忽略。
# 书的技巧：fa、fb 都居中排列，FFT 前【不移位】——两个函数携带同一线性
# 相位，乘积相位对应"整周期平移"，模 L 不可见；但逆变换结果停在"移位
# 排列"（峰值在 index 0），所以要 fftshift 居中。只移位其中一个函数，
# 卷积结果就会整体错位 L/2（本脚本有反例演示）。
# MATLAB/Python 差异：MATLAB 要 .* 与 .^ 表示逐元素，忘点号不报错但结果
# 离谱；numpy 的 * 与 ** 天然逐元素，矩阵乘法反而要另写 @ / np.matmul
# ——两边的"默认值"方向恰好相反。
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "common"))
from base import coords   # 公共积木：式 (2.21)/(2.22) 的网格坐标（ch2 起）

wa, wb = 0.3, 0.2         # 高斯宽度 [exp(-pi) 半径] (m)
L, M = 2.0, 200           # 边长 (m)、样本数
dx = L / M                # 采样间隔 (m)
x, _ = coords(L, M)       # x 坐标（式 2.21）
fa = np.exp(-np.pi * x ** 2 / wa ** 2)   # 高斯 a
fb = np.exp(-np.pi * x ** 2 / wb ** 2)   # 高斯 b

# ---------- FFT 卷积（书 3.6 节，输入不移位） ----------
Fa = np.fft.fft(fa)                       # 变换 fa
Fb = np.fft.fft(fb)                       # 变换 fb
F0 = Fa * Fb                              # 逐点相乘（numpy 的 * 就是 MATLAB 的 .*）
f0 = np.fft.ifft(F0) * dx                 # 逆变换；卷积定标只乘一次 dx（不是 dx^2）
f = np.fft.fftshift(f0.real)              # 居中（书图 3.9b；虚部是机器精度残渣）

# ---------- 解析卷积（习题 3.2） ----------
# exp(-pi*x^2/w^2) <-> w*exp(-pi*w^2*fx^2)（表 1.3 高斯对）。频域相乘：
# wa*wb*exp(-pi*(wa^2+wb^2)*fx^2)，逆变换回去：
#   c(x) = wa*wb/sqrt(wa^2+wb^2) * exp(-pi*x^2/(wa^2+wb^2))
c_an = wa * wb / np.sqrt(wa ** 2 + wb ** 2) * np.exp(-np.pi * x ** 2 / (wa ** 2 + wb ** 2))

print(f"解析卷积峰值 = {c_an.max():.6f}；FFT 卷积峰值 = {f.max():.6f}")
print(f"FFT 卷积与解析最大差异 = {np.max(np.abs(f - c_an)):.2e}"
      "（高斯衰减极快：离散求和与连续积分之差达机器精度）")

# ---------- 反例：只移位一个函数 -> 卷积整体错位 ----------
fa_s = np.fft.ifftshift(fa)               # 只把 fa 挪到移位排列
f_wrong = np.fft.fftshift((np.fft.ifft(np.fft.fft(fa_s) * np.fft.fft(fb)) * dx).real)
iw = int(np.argmax(f_wrong))
print(f"只移位 fa：卷积峰出现在 x = {x[iw]:.2f} m（应为 0 -> 整体错位 L/2）")

# ---------- 画图：复现书图 3.9 + 解析对比 ----------
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
axes[0].plot(x, fa, label=r"$f_a$ ($w_a$=0.3)")
axes[0].plot(x, fb, "--", label=r"$f_b$ ($w_b$=0.2)")
axes[0].set_xlabel("x (m)"); axes[0].set_title("functions (Fig. 3.9a)")
axes[0].legend(fontsize=8)
axes[1].plot(x, f, label="FFT")
axes[1].plot(x, c_an, ":", label="analytic")
axes[1].set_xlabel("x (m)")
axes[1].set_title("convolution: FFT vs analytic (Fig. 3.9b + Ex. 3.2)")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(os.path.dirname(os.path.abspath(__file__)),
                         "out_conv_example.png"), dpi=150)
print("已保存 out_conv_example.png")


## 习题

`ch3_exercises.py`

In [ ]:
# ===== 书 3.9 节习题 3.1（三角函数）与 3.3（圆函数）参考解答 =====
# 3.1：底半宽 0.1 m 的三角函数，L = 2、M = 200。书提示的编码：
#       T=1-abs(x); ask=abs(x)<=1; out=T.*mask;
#     其中 ask 与 mask 是同一个变量（原书笔误，两个名字混用，本文件统一
#     写成 mask）。按 3.1(b) 的尺度（底半宽 w = 0.1）缩放后，
#     解析频谱：tri(x/w) <-> w sinc^2(w fx)，w = 0.1 -> 0.1 sinc^2(0.1 fx)。
# 3.3：半径 0.015 m 的圆盘，L = 0.2、M = 200（dx = 0.001，半径恰 15 个样本）。
#     频谱 = jinc 形：R*J1(2*pi*R*rho)/rho，峰值 = 面积 pi*R^2 ≈ 7.07e-4。
#     （半径约定见第 1 章勘误 1：本解按"半径 = R"的 Goodman 约定，
#      与表 1.3 的 circ 变换对一致。）
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "common"))
from base import coords, sinc, tri, circle, jinc   # 公共积木（ch2 起）

# ---------- 习题 3.1：三角函数 ----------
L, M = 2.0, 200
dx = L / M
x, fx = coords(L, M)
w = 0.1                                # 底半宽 (m)
T = 1 - np.abs(x) / w                  # 书提示第 1 行（按习题尺度缩放）
mask = np.abs(x) <= w                  # 书提示第 2 行（原书写成 ask——笔误）
tri1 = T * mask                        # 第 3 行：边界 |x|=w 处 T=0，无半值歧义
print(f"[3.1] 与 base.tri(x/w) 最大差异 = "
      f"{np.max(np.abs(tri1 - tri(x / w))):.1f}（同一函数的两种写法）")

TRI = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(tri1))) * dx   # 居中 FFT × dx
TRI_an = w * sinc(w * fx) ** 2         # tri(x/w) <-> w sinc^2(w fx)

print(f"[3.1] 三角函数 FFT 峰值 = {np.abs(TRI).max():.6f}"
      f"（解析 w = 0.1 = 三角形面积）")
in_main = np.abs(fx) <= 30
err_tri = np.max(np.abs(np.abs(TRI[in_main]) - TRI_an[in_main]))
print(f"[3.1] FFT 与解析最大差异（|fx|<=30 内）= {err_tri:.2e}")

# ---------- 习题 3.3：圆函数 ----------
L2, M2 = 0.2, 200
dx2 = L2 / M2
x2, fx2 = coords(L2, M2)
y2 = x2
X, Y = np.meshgrid(x2, y2)             # 'xy'：行 <-> y
R = 0.015                              # 圆半径 (m) = 15 个样本

# 两种编码只差"圆周上的样本"：书风格（<=，边界取 1）vs base.circle（<）
circ_le = (np.hypot(X, Y) <= R).astype(float)   # 习题 3.3(a) 的书风格编码
circ_lt = circle(X, Y, R)                       # base 参考实现（严格小于）
n_bd = int(np.sum((np.hypot(X, Y) <= R).astype(int)) - np.sum((np.hypot(X, Y) < R).astype(int)))
print(f"[3.3] 半径 R = {R} m，两编码相差的圆周样本数 = {n_bd}"
      f"（对本例网格：{n_bd} 个）")

G_le = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(circ_le))) * dx2 ** 2
G_lt = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(circ_lt))) * dx2 ** 2
row = M2 // 2                          # fy=0 剖面所在行
J_an = jinc(fx2, R)                    # 半径 R 圆盘的解析频谱（Goodman 约定锚点）
print(f"[3.3] 圆盘 FFT 峰值 = {np.abs(G_le).max():.6e}"
      f"（解析面积 πR² = {np.pi * R ** 2:.6e}）")
in_main2 = np.abs(fx2) <= 30
err_le = np.max(np.abs(np.abs(G_le[row, :][in_main2]) - np.abs(J_an[in_main2])))
err_lt = np.max(np.abs(np.abs(G_lt[row, :][in_main2]) - np.abs(J_an[in_main2])))
print(f"[3.3] fy=0 剖面与 jinc 最大差异（|fx|<=30 内）："
      f"边界含圆周 {err_le:.2e}；不含 {err_lt:.2e}（相对峰值 ~1%，圆盘阶梯离散的正常水平）")

# ---------- 画图 ----------
fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.0))

axes[0, 0].plot(x, tri1, "o-", ms=4, mfc="none")
axes[0, 0].set_xlim(-0.3, 0.3); axes[0, 0].set_ylim(0, 1.2)
axes[0, 0].set_xlabel("x (m)")
axes[0, 0].set_title("Ex 3.1: triangle, half-base 0.1 m")

axes[0, 1].plot(fx, np.abs(TRI), label="discrete")
axes[0, 1].plot(fx, TRI_an, ":", label="analytic")
axes[0, 1].set_xlim(-60, 60); axes[0, 1].set_xlabel("fx (cyc/m)")
axes[0, 1].set_title(r"Ex 3.1: $|FFT|$ vs $w\,\mathrm{sinc}^2(w f_x)$")
axes[0, 1].legend(fontsize=8)

axes[1, 0].imshow(circ_le, extent=[x2[0], x2[-1], y2[0], y2[-1]],
                  origin="lower", cmap="gray")
axes[1, 0].set_xlabel("x (m)"); axes[1, 0].set_ylabel("y (m)")
axes[1, 0].set_title("Ex 3.3: circle, R = 0.015 m")

axes[1, 1].plot(fx2, np.abs(G_le[row, :]), "o", ms=4, mfc="none", label="discrete")
axes[1, 1].plot(fx2, np.abs(J_an), ":", label="analytic")
axes[1, 1].set_xlim(-60, 60); axes[1, 1].set_xlabel("fx (cyc/m)")
axes[1, 1].set_title(r"Ex 3.3: slice vs jinc $R J_1(2\pi R\rho)/\rho$")
axes[1, 1].legend(fontsize=8)

for ax in axes.flat:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(os.path.dirname(os.path.abspath(__file__)),
                         "out_exercises.png"), dpi=150)
print("已保存 out_exercises.png")


## fft2 · example

`ch3_fft2_example.py`

In [ ]:
# ===== 书 3.7 节：二维矩形函数、meshgrid、图像显示与二维 FFT =====
# 参数：wx = 0.1 m、wy = 0.05 m 的矩形（0.2 × 0.1 m），L = 2 m、M = 200/边
# -> dx = 0.01 m，200×200 阵列对应 2 × 2 m。
# MATLAB/Python 差异：
#   - np.meshgrid 默认 indexing='xy' 与 MATLAB meshgrid 一致（X 的行是 x
#     的副本、Y 的列是 y 的副本）；写 'ij' 两维会互换；
#   - MATLAB g(M/2+1,:) 选第 M/2+1 行 = y=0 的行（行 <-> y！）；Python
#     0-based 是 g[M//2, :]；
#   - plt.imshow 默认 y 向下，origin='lower' 翻转（MATLAB 用 axis xy）；
#   - 二维必须 fft2（np.fft.fft2）：一维 fft 作用于二维阵列只变换一行行
#     向量，得到"条纹"状错误结果（书 3.8 节原话）。
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  （注册 3D 投影，新版本可省）

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "common"))
from base import coords, sinc   # 公共积木（ch2 起）


def rect_book(x):
    """书 3.1 节的 rect 编码：abs(x) <= 1/2 -> 1（见 ch3_fft_example.py 注释）。"""
    return (np.abs(x) <= 0.5).astype(float)


wx, wy = 0.1, 0.05        # 矩形 x/y 半宽 (m)
L, M = 2.0, 200           # 边长 (m)、每边样本数
dx = L / M                # 采样间隔 (m)
x, _ = coords(L, M)       # x 坐标
y = x                     # y 坐标
X, Y = np.meshgrid(x, y)  # X 的行 = x 的副本；Y 的列 = y 的副本
g = rect_book(X / (2 * wx)) * rect_book(Y / (2 * wy))   # 二维矩形（可分离乘积）

# ---------- 二维 FFT ----------
# 书：g0=fftshift(g); G0=fft2(g0)*dx^2; G=fftshift(G0)——偶数 M 下
# fftshift 与 ifftshift 等价；本文件按第 2 章的严谨配对写 ifftshift。
G0 = np.fft.fft2(np.fft.ifftshift(g)) * dx ** 2   # 二维 FFT × dx^2 定标
G = np.fft.fftshift(G0)                           # 居中
_, fx = coords(L, M)                              # 频率坐标（式 2.22；coords 首个返回值是空域 x！）
fy = fx                                           # 两维相同

# ---------- 数值检查 ----------
row = M // 2                       # y=0 的行（MATLAB g(M/2+1,:)）
# 书 rect 编码（边界取 1）下，x 向 |X|<=0.1 命中 21 个样本、y 向 11 个：
# wx=0.1、wy=0.05 都是"偶数样本宽"，按 3.1 节的约定各多给一个边界样本。
nx = int(rect_book(x / (2 * wx)).sum())   # 21
ny = int(rect_book(y / (2 * wy)).sum())   # 11
# 离散精确谱（Dirichlet 核）：FFT 剖面应与它机器精度重合
with np.errstate(invalid="ignore", divide="ignore"):
    D = np.sin(nx * np.pi * fx * dx) / np.sin(np.pi * fx * dx)
D[fx == 0] = nx                                   # 0/0 极限
slice_exact = dx ** 2 * ny * np.abs(D)
slice_an = 4 * wx * wy * np.abs(sinc(2 * wx * fx))  # 连续矩形解析谱
in_main = np.abs(fx) <= 25
err_exact = np.max(np.abs(np.abs(G[row, :][in_main]) - slice_exact[in_main]))
err_an = np.max(np.abs(np.abs(G[row, :][in_main]) - slice_an[in_main]))
print(f"x/y 向值为 1 的样本数 = {nx} x {ny}（边界取 1 的奇数样本约定）")
print(f"DC 样本 |G(0,0)| = {abs(G[row, row]):.6f} = 离散面积 "
      f"{nx}·{ny}·dx² = {nx * ny * dx ** 2:.4f}")
print(f"  （连续矩形面积 4wx·wy = {4 * wx * wy:.3f}；书图 3.11(b) 峰值 ~0.023 "
      "就是离散面积，")
print("    不是混叠——wx、wy 恰好都是偶数样本宽，rect 各多给一个边界样本）")
print(f"fy=0 剖面 vs 离散精确谱（Dirichlet）最大差异 = {err_exact:.2e}（机器精度）")
print(f"fy=0 剖面 vs 连续解析谱最大差异 = {err_an:.4f}"
      "（DC 处 0.0231 vs 0.020 + 零点错位）")

# ---------- 显示：图像 + 切片 + 曲面 + 频域剖面（书图 3.10/3.11） ----------
fig = plt.figure(figsize=(12.5, 8.4))

ax1 = fig.add_subplot(2, 2, 1)
ax1.imshow(g, extent=[x[0], x[-1], y[0], y[-1]],
           origin="lower", cmap="gray")          # origin='lower' = MATLAB axis xy
ax1.set_xlabel("x (m)"); ax1.set_ylabel("y (m)")
ax1.set_title("g(x,y): 2D rect (Fig. 3.10a)")

ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(x, g[row, :], "o-", ms=3, color="crimson", mfc="none")  # y=0 行切片
ax2.set_xlim(-1, 1); ax2.set_ylim(0, 1.2)
ax2.set_xlabel("x (m)")
ax2.set_title("x-slice through g[M//2,:] (Fig. 3.10b)")

ax3 = fig.add_subplot(2, 2, 3, projection="3d")
FXf, FYf = np.meshgrid(fx, fy)                     # 曲面用频率网格
ax3.plot_surface(FXf, FYf, np.abs(G), cmap="gray",
                 rstride=2, cstride=2, linewidth=0)
ax3.set_xlabel("fx (cyc/m)"); ax3.set_ylabel("fy (cyc/m)")
ax3.set_title(r"$|G(f_X,f_Y)|$ surface (Fig. 3.11a)")

ax4 = fig.add_subplot(2, 2, 4)
ax4.plot(fx, np.abs(G[row, :]), label="FFT")
ax4.plot(fx, slice_exact, ":", lw=1, label="discrete exact (Dirichlet)")
ax4.plot(fx, slice_an, "--", lw=1, label=r"$4w_xw_y\,\mathrm{sinc}(2w_x f_x)$")
ax4.set_xlim(-50, 50); ax4.set_xlabel("fx (cyc/m)")
ax4.set_title("fx slice: FFT vs two references (Fig. 3.11b)")
ax4.legend(fontsize=8)

for ax in (ax1, ax2, ax4):
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(os.path.dirname(os.path.abspath(__file__)),
                         "out_fft2_example.png"), dpi=150)
print("已保存 out_fft2_example.png")


## fft · example

`ch3_fft_example.py`

In [ ]:
# ===== 书 3.2-3.5 节：fft_example 脚本（建向量 -> 移位 -> FFT 定标 -> 解析对比） =====
# 书把这几节写成一个逐步加长的脚本 fft_example.m（原书第 1-58 行），本文件
# 按同样的参数与步骤复现书图 3.2-3.8，并附数值检查。
# 参数（书 3.2 节）：矩形半宽 w = 0.055 m（全宽 0.11 m）、边长 L = 2 m、
# 样本数 M = 200 -> 采样间隔 dx = 0.01 m。
# MATLAB/Python 差异：
#   - numpy 索引 0-based：居中排列的零坐标在 index M//2（本例 100），
#     MATLAB 是 index M/2+1（本例 101）；
#   - 书用 fftshift 做 FFT 前移位（偶数 M 下与 ifftshift 等价）；本文件按
#     第 2 章的严谨配对写 ifftshift -> fft -> fftshift，奇数 M 也不出错；
#   - MATLAB 的 plot '-o'、axis([...]) 对应 plt.plot(..., 'o-') 与
#     plt.xlim/plt.ylim。
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "common"))
from base import rect, coords, sinc   # 公共积木：表 1.2 函数与网格约定（ch2 起）


def rect_book(x):
    """书 3.1 节的 rect 编码：abs(x) <= 1/2 -> 1（边界取 1，不取半值）。

    表 1.2 的解析定义（base.rect）在边界取半值 1/2；两者只在恰好命中
    边界的样本上不同。书 3.1 节原话：半值会被解读成"斜坡"，不如单点
    跳变；代价是矩形全宽总由奇数个样本组成（请求偶数宽会多给一个样本），
    所以全书例子故意用 0.055 这类非整半宽。
    """
    return (np.abs(x) <= 0.5).astype(float)


# ---------- 3.2 创建向量 ----------
w = 0.055          # 矩形半宽 (m)。故意不取 0.05：见"奇数样本"特性
L = 2.0            # 向量边长 (m)
M = 200            # 样本数
dx = L / M         # 采样间隔 (m)
x, fx = coords(L, M)           # 式 (2.21)/(2.22)：x ∈ [-1, 0.99]，fx ∈ [-50, 49.5]
f = rect_book(x / (2 * w))     # 采样矩形：|x| <= 0.055 的样本为 1（11 个）

print(f"x 范围: [{x[0]:.2f}, {x[-1]:.2f}]，共 {len(x)} 个样本")
print(f"f==1 的样本数 = {int(f.sum())}（全宽 0.11 m / dx 0.01 m = 11，奇数）")
print(f"坐标 x=0 的 index = {int(np.argmin(np.abs(x)))}"
      f"（Python 0-based；MATLAB 是 {M // 2 + 1} = M/2+1）")

# 两种 rect 约定对照：本例网格上完全相同（边界 x=±0.055 不在网格上）
diff = np.max(np.abs(f - rect(x / (2 * w))))
print(f"rect_book 与表 1.2 的 base.rect 在本网格最大差异 = {diff:.1f}"
      f"（边界样本不在网格 -> 相同；若请求偶数全宽就会多出一个样本）")

# ---------- 3.3 为 FFT 移位 ----------
f0 = np.fft.ifftshift(f)   # 居中 -> 移位排列（零坐标样本挪到 index 0）
# 移位后中心样本在 index 0，5 个样本在"右"（index 1..5），其余 5 个在
# "左"——挂在向量末尾（index 195..199），对应书图 3.5。
# 不移位直接 fft 的后果：相当于对平移了 -L/2 的函数做变换，频域多出一线性
# 相位因子 exp(+j*2*pi*f*L/2)（位移定理）——幅度不变、相位全错。

# ---------- 3.4 计算 FFT 并定标 ----------
F0 = np.fft.fft(f0) * dx   # FFT 并乘 dx 回连续量纲（式 2.19 + 黎曼和因子）
F = np.fft.fftshift(F0)    # 居中排列
# fx 已由 coords 给出（式 2.22）：-50 : 0.5 : 49.5

# ---------- 3.5 与解析结果对比 ----------
F_an = 2 * w * sinc(2 * w * fx)   # rect(x/2w) <-> 2w sinc(2w fx)（表 1.3，a = 2w）
# np.sinc 与书用的 MATLAB 内置 sinc（信号处理工具箱）同定义：sin(pi u)/(pi u)

# ---------- 数值检查 ----------
ic = int(np.argmax(np.abs(F)))
print(f"DC 样本 |F(fx=0)| = {abs(F[ic]):.6f}（= 矩形面积 2w = {2 * w:.2f}，中心纵坐标定理）")
in_main = np.abs(fx) <= 25
err_all = np.max(np.abs(np.abs(F) - np.abs(F_an)))
err_main = np.max(np.abs(np.abs(F[in_main]) - np.abs(F_an[in_main])))
print(f"|FFT|-|解析| 最大差异：全频带 {err_all:.4f}；|fx|<=25 内 {err_main:.2e}")
print("  （边缘差异 = 周期延拓带来的混叠，书 3.5 节原话'FFT 边缘略高'）")
print(f"sinc 零点间隔 1/(2w) 折合样本数：L/(2w) = {L / (2 * w):.2f}（书说'约每 18 个点'）")
print(f"峰值处相位 = {np.angle(F[ic]):.2e}（实偶函数 -> 相位只有 0 与 ±π，±π 同一相位 mod 2π）")

# ---------- 画图：复现书图 3.3-3.8 ----------
fig, axes = plt.subplots(2, 3, figsize=(12.5, 6.8))

axes[0, 0].plot(x, f, "o-", ms=4, color="crimson", mfc="none")     # 书图 3.3
axes[0, 0].set_xlim(-0.2, 0.2); axes[0, 0].set_ylim(0, 1.5)
axes[0, 0].set_xlabel("x (m)")
axes[0, 0].set_title("f = rect(x/(2w)) samples (Fig. 3.3)")

axes[0, 1].plot(np.arange(1, M + 1), f, "o-", ms=4, mfc="none")    # 书图 3.4（1-based 展示）
axes[0, 1].set_xlim(80, 120); axes[0, 1].set_ylim(0, 1.5)
axes[0, 1].set_xlabel("index (MATLAB 1-based)")
axes[0, 1].set_title("vs index: center at 101 = M/2+1 (Fig. 3.4)")

axes[0, 2].plot(np.arange(1, M + 1), f0, "o-", ms=4, mfc="none")   # 书图 3.5
axes[0, 2].set_xlim(0, 200); axes[0, 2].set_ylim(0, 1.5)
axes[0, 2].set_xlabel("index (MATLAB 1-based)")
axes[0, 2].set_title("after shift: center at index 1 (Fig. 3.5)")

axes[1, 0].plot(np.abs(F0))                                        # 书图 3.6(a)
axes[1, 0].set_xlabel("index")
axes[1, 0].set_title(r"$|F_0|$ vs index (Fig. 3.6a)")

axes[1, 1].plot(np.angle(F0))                                      # 书图 3.6(b)
axes[1, 1].set_xlabel("index")
axes[1, 1].set_title("phase vs index: only 0 and $\\pm\\pi$ (Fig. 3.6b)")

axes[1, 2].plot(fx, np.abs(F), fx, np.abs(F_an), ":")              # 书图 3.8(a)
axes[1, 2].set_xlabel("fx (cyc/m)")
axes[1, 2].set_title("centered |F| vs analytic (Fig. 3.8a)")
axes[1, 2].legend(["discrete", "analytic"], fontsize=8)

for ax in axes.flat:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(os.path.dirname(os.path.abspath(__file__)),
                         "out_fft_example.png"), dpi=150)
print("已保存 out_fft_example.png")
